<a href="https://colab.research.google.com/github/Nijimbere722/lab-4-llm-decision-support/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4



## Part 1.1

In [1]:
import os
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")
from openai import OpenAI
client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"
print("Client ready.")

Client ready.


In [2]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response


# Test call
resp = ask_llm("What are three things a microfinance loan officer should look for in a loan application letter?")
answer_text = resp.choices[0].message.content
print(answer_text)
print("\n--- Token usage ---")
print(resp.usage)

When reviewing a loan application letter, a microfinance loan officer should look for the following three things:

1. **Clear Business Plan and Use of Funds**: The loan officer should look for a clear description of the business or project for which the loan is being sought, including the specific use of funds. This could include details on how the loan will be used to purchase equipment, inventory, or expand operations. The loan officer wants to see that the borrower has a well-thought-out plan for using the loan to generate income and repay the loan.

2. **Creditworthiness and Repayment Ability**: The loan officer should assess the borrower's creditworthiness and ability to repay the loan. This includes reviewing the borrower's credit history, income, and expenses to determine if they have a stable financial situation and a reliable source of income to make loan payments. The loan officer may also look for collateral or a guarantor to secure the loan.

3. **Realistic Financial Projec

1. System vs. user roles:
The system role sets the model's overall behavior, persona, and rules for the whole conversation. It's set once and stays in effect. The user role is the actual input or question for that specific turn.

* Example for system: "You are an assistant to a microfinance loan officer. Be factual and neutral, and never invent details that ar not in the letter."
Example for user: "Summarize this loan application:"letter text"

2. What is a token:
a small chunk of text such as a word, part of a word, a number, or a punctuation mark.

* API providers bill per token rather than per request because the actual computational cost is driven by how much text the model has to read and generate, not by how many times you called the API. A one-word request and a 500-word request are both "one request," but the second one takes far more compute, so billing by token ties the cost directly to the actual work done.

## Part 1.2

In [3]:
question = "Suggest a name for a savings product for market traders in Accra."

low_temp_answers = [ask_llm(question, temperature=0.0).choices[0].message.content for _ in range(5)]
high_temp_answers = [ask_llm(question, temperature=1.2).choices[0].message.content for _ in range(5)]

print("=== Temperature = 0.0 ===")
for i, a in enumerate(low_temp_answers, 1):
    print(f"\n[{i}] {a}")

print("\n\n=== Temperature = 1.2 ===")
for i, a in enumerate(high_temp_answers, 1):
    print(f"\n[{i}] {a}")

=== Temperature = 0.0 ===

[1] Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, and this name immediately conveys that the product is tailored to market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth, which is a key goal for market traders.
3. **Sika Souce**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. This name incorporates a local touch and could resonate with market traders.
4. **Market Booster**: This name suggests that the savings product will help market traders boost their businesses and achieve their financial goals.
5. **Adanfo Save**: "Adanfo" means "friends" or "partners" in the Akan language. This name conveys a sense of community and partnership, which could be appealing to market traders.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian term that means "honest" or "transparent". This name emphasizes 

* At temperature 0.0, the five answers were nearly identical across runs, the same names ("Makola Save," "Sika Su," "Kokroko Savings") reappeared almost word-for-word each time, and two runs were exact duplicates. At temperature 1.2, the five answers varied much more different Ghanaian languages drawn on, different name structures, and more unusual suggestions. For the loan decision-support system, low temperature (0–0.2) is the right choice, because summarizing and extracting data from a loan letter needs to be consistent and factual , a loan officer shouldn't get a different summary of the same letter each time it's processed.

## Section 2

In [6]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")
print("\n L002\n", LETTERS["L002"])
print("\nL006 \n", LETTERS["L006"])

6 letters loaded.

 L002
 Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.

L006 
 Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.


## Section 3

### Part 3.1

In [7]:
SUMMARY_PROMPT_V1 = "Summarize this: {letter}"

for lid in ["L002", "L006"]:
    out = ask_llm(SUMMARY_PROMPT_V1.format(letter=LETTERS[lid]))
    print(f" {lid} (V1)\n{out.choices[0].message.content}\n")

 L002 (V1)
Kwame Boateng, a commercial driver in Kumasi, is seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season, and is willing to repay the loan when his finances allow. He doesn't have collateral to offer at the moment.

 L006 (V1)
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no experience or collateral, but claims to be "business-minded" and promises to repay the loan within a year when his businesses become successful, relying on his personal trustworthiness as assurance.



In [8]:
SUMMARY_SYSTEM_PROMPT_V2 = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Summarize loan application letters factually and neutrally in 3-4 sentences. "
    "Only include information explicitly stated in the letter. "
    "Do not invent, infer, or embellish any detail not present in the source text. "
    "Do not offer an opinion on whether the loan should be approved."
)

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter}"

def summarize_letter(letter_text):
    resp = ask_llm(
        SUMMARY_PROMPT_V2.format(letter=letter_text),
        system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
        temperature=0,
    )
    return resp.choices[0].message.content

for lid in ["L002", "L006"]:
    out = summarize_letter(LETTERS[lid])
    print(f" {lid} (V2) \n{out}\n")

 L002 (V2) 
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that business has been slow, but expects it to improve after the festive season. He does not currently have collateral to offer, but is requesting assistance as soon as possible.

 L006 (V2) 
Kofi is applying for a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He is 22 years old and claims to be "full of energy" and "business minded" as stated by his friends. Kofi has not yet started any of these businesses. He intends to repay the loan in one year and does not have collateral, but describes himself as trustworthy.



1. Difference between V1 and V2:

* V1 ( "summarize this" prompt) added small opinions that weren't in the letter. For L006, it said Kofi "claims" to be business-minded and is "relying on" his trustworthiness,these words sound doubtful, like the summary doesn't fully believe him, even though the letter never said that.

* V2 (with clear instructions to be factual and neutral) fixed this. It still mentions that Kofi says he is "business minded," but it clearly says this came from his friends ("as stated by his friends") instead of sounding doubtful. It reports what the letter says without adding a hidden opinion.

2. Why "no invented details" matters:

* A loan officer may only read the summary and not the full letter. If the summary adds or changes small details, the officer could make a decision based on wrong information. When an AI adds information that was never actually said, this is called hallucination. It is a serious problem for something like a loan system, because it could affect a real financial decision.